In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
from pathlib import Path
from langchain_core.documents import Document
import re  # For timestamp parsing
import os

folder_path = r"C:\Users\sohai\Desktop\LAB\Final_Project\Final_Files"
documents = []

for txt_file in Path(folder_path).glob("**/*.txt"):
    with open(txt_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Extract timestamps (WEBVTT format: 00:01:23.456 --> 00:01:25.789)
    timestamps = re.findall(r'(\d{2}:\d{2}:\d{2}\.\d{3}) --> (\d{2}:\d{2}:\d{2}\.\d{3})', content)
    
    doc = Document(
        page_content=content,
        metadata={
            "source": str(txt_file),
            "type": "txt",
            "file_name": txt_file.name,
            "num_segments": len(timestamps),
            "first_timestamp": timestamps[0][0] if timestamps else "00:00:00.000",
            "last_timestamp": timestamps[-1][1] if timestamps else "00:00:00.000",
            "duration_estimate": "Unknown"  # Could calculate
        }
    )
    documents.append(doc)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ New package

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i

for doc in chunks:
    doc.metadata["video_id"] = doc.metadata.get("file_name").split("/")[-1].replace(".txt", "")

c:\Users\sohai\Desktop\LAB\Final_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "rag"
if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name,dimension=1536,metric="cosine",spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)
embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))
vectorstore = PineconeVectorStore(embedding=embeddings, index=index)
vectorstore.add_documents(chunks)

['50d2db8c-5082-47cd-b266-0d121420b403',
 '1db045a3-4854-4c70-8f65-0fd7c46c6055',
 'd12804dc-6646-4824-a8e2-5a4b643c1b2e',
 '7e817900-a533-4b6e-91f3-3f95c5e7ac31',
 'ce380c41-4a71-4123-92e8-9a7d73bcf7e8',
 '75de73b1-56c2-45c2-8017-331c79b97077',
 '36f29603-a059-4a5b-aa2c-279973d88e31',
 '983253bc-0118-461e-b0d7-b952c65f71f2',
 'bdbc1f52-e5a2-41bb-b94c-5f1412470cab',
 '46d2d97e-b5f2-418e-b99a-848b55b263b4',
 '76335ba9-832b-4c3f-b904-143f5da22ea9',
 '4a32b9c7-6727-4f68-bf07-80d9a3155698',
 '24b2fd30-a75c-40c3-b53d-463488766b8f',
 '188b0c7f-712b-4084-bf36-f12a544577d8',
 '144c6d8d-077c-4d50-b455-82d07b74b62f',
 '227d4388-dec5-4db5-a416-e752f025caca',
 '365edb14-0d80-46d3-9626-5bbe9f99570f',
 '7fcd1b81-658b-41c3-86c7-c7f66c4c4beb',
 '8df29a39-fe64-4682-afd7-62b0b7b9cb1e',
 '3baf051e-3e8d-4fc1-9c39-41d2433394f4',
 'd6365937-72ce-4e4c-a819-a65d38d1a582',
 '47e0f79e-da66-40ad-a59a-e09ed8dfe88b',
 '858e958b-7a0d-46a9-bd55-a3c643213f5d',
 '4a296c6a-d260-47cf-9531-8a92de1b9e1d',
 '4ae918f0-8e2d-

In [5]:
import os
import uuid
import gradio as gr
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import socket

# ── 1. LLM & Retriever ────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

# ── 2. PromptTemplate ─────────────────────────────────────────────────────
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI assistant answering questions using a knowledge base.
Use ONLY the information provided in the context below.

Context: {context}

Question: {question}

If the answer is not in the context, say: "I could not find the answer in the provided documents."
Answer:
"""
)

# ── 3. RAG search function ────────────────────────────────────────────────
def search_documents(question: str):
    docs = retriever.invoke(question)
    context_with_metadata = "\n\n".join([
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in docs
    ])
    final_prompt = rag_prompt.format(
        context=context_with_metadata,
        question=question
    )
    response = llm.invoke(final_prompt)
    return response.content

# ── 4. Tool ───────────────────────────────────────────────────────────────
@tool
def rag_search(question: str) -> str:
    """Search the vector database and answer questions about neural networks."""
    return search_documents(question)

# ── 5. Memory + Agent ─────────────────────────────────────────────────────
memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[rag_search],
    checkpointer=memory,
    prompt="You are a helpful AI assistant that answers questions about neural networks "
           "using the rag_search tool. Always use the tool to find answers. "
           "Remember the conversation history and refer to previous answers when relevant."
)

# ── 6. Chat function with memory ──────────────────────────────────────────
def chat(question: str, history: list, session_id: str) -> tuple:
    if not question.strip():
        return history, "", session_id

    # ✅ Each browser session gets its own memory thread
    config = {"configurable": {"thread_id": session_id}}

    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config                        # ✅ memory tied to session_id
        )
        answer = result["messages"][-1].content

        # Fetch sources separately to display in UI
        docs = retriever.invoke(question)
        sources = "\n\n".join([
            f"📄 **Chunk {i+1}** — `{doc.metadata.get('source', 'unknown')}`\n"
            f"{doc.page_content[:300]}..."
            for i, doc in enumerate(docs)
        ])
        full_response = f"{answer}\n\n---\n**📚 Retrieved Sources:**\n{sources}"

    except Exception as e:
        full_response = f"❌ Error: {str(e)}"

    history = history + [
        {"role": "user",      "content": question},
        {"role": "assistant", "content": full_response}
    ]
    return history, "", session_id

# ── 7. Clear function — resets memory by generating new session ID ─────────
def clear_chat():
    new_session_id = str(uuid.uuid4())           # ✅ new thread = fresh memory
    return [], "", new_session_id



C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\480239116.py:54: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [6]:
import os
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
import numpy as np
import datetime

# ── 1. Set LangSmith env vars BEFORE anything else ────────────────────────
os.environ["LANGCHAIN_TRACING_V2"]  = "true"
os.environ["LANGCHAIN_PROJECT"]     = "ragas-rag-eval"
os.environ["LANGCHAIN_API_KEY"]     = os.getenv("LANGSMITH_API_KEY")  

# ── 2. Load contexts ───────────────────────────────────────────────────────
def load_context(filename):
    path = os.path.join(r"C:\Users\sohai\Desktop\LAB\Final_Project\Final_Files", filename)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# ── 3. Build dataset ───────────────────────────────────────────────────────
data = {
    "question": [
        "What is hidden layer in neural networks?",
        "What is activation function in neural networks?",
        "What is gradient descent in neural networks?",
        "What is neural network?",
        "What is backpropagation?"
    ],
    "answer": [
        "A hidden layer in neural networks refers to the layers of neurons that exist between the input layer and the output layer.",
        "An activation function is a mathematical function that determines the output of a neuron based on its input.",
        "Gradient descent is a method used to minimize the cost function by iteratively adjusting the model's parameters.",
        "A neural network is inspired by the brain, consisting of interconnected neurons that hold numerical values between zero and one.",
        "Backpropagation evaluates the loss function and determines how much each neuron contributed to errors."
    ],
    "contexts": [
        [load_context("f4.txt")],
        [load_context("f1.txt")],
        [load_context("f2.txt")],
        [load_context("f1.txt")],
        [load_context("f8.txt")]
    ],
    "reference": [
        "A hidden layer is any intermediate layer of neurons between the input and output layers.",
        "Activation functions are non-linear mathematical operations applied to a neuron's output.",
        "Gradient descent minimizes loss by iteratively adjusting network weights and biases.",
        "Neural networks are machine learning systems inspired by the human brain.",
        "Backpropagation calculates the gradient of the loss function to adjust weights."
    ]
}

ragas_dataset = Dataset.from_dict(data)

# ── 4. LLM & Embeddings ────────────────────────────────────────────────────
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

import asyncio

try:
    loop = asyncio.get_event_loop()
except RuntimeError:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

# ── 5. Run RAGAS — LangSmith auto-traces via env vars ─────────────────────
print("Running RAGAS evaluation...")
results = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=LangchainLLMWrapper(llm),
    embeddings=LangchainEmbeddingsWrapper(embeddings)
)

# ── 6. Print results ───────────────────────────────────────────────────────
print("\n── RAGAS Scores ──")
print(results)
ragas_scores = {}
print("\n── Per-metric Breakdown ──")

metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
for metric in metric_names:
    score = results[metric]
    avg = float(np.mean(score)) if isinstance(score, list) else float(score)
    ragas_scores[metric] = avg
    print(f"  {metric}: {avg:.4f}")


print("\n── Per-question Scores ──")
df = results.to_pandas()
#print(df.to_string())
numeric_cols = df.select_dtypes(include='number').columns
print(df[numeric_cols].to_string())

# ── 7. Manually log aggregate scores as a single run to LangSmith ─────────
import uuid
import datetime

client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

# Prepare scores
scores = {
    metric: float(np.mean(results[metric])) if isinstance(results[metric], list)
            else float(results[metric])
    for metric in metric_names
}

# Use a fixed run ID
run_id = uuid.uuid4()

client.create_run(
    id=run_id,                          # ✅ pass ID explicitly
    name="ragas-evaluation",
    run_type="chain",
    project_name="ragas-rag-eval",
    inputs={
        "dataset_size": len(ragas_dataset),
        "metrics": metric_names,
        "model": "gpt-4o-mini"
    },
    outputs=scores,
    start_time=datetime.datetime.utcnow(),
    end_time=datetime.datetime.utcnow()  # ✅ pass end_time in create_run directly
)

print(f"\n✅ Scores logged to LangSmith: {scores}")

# ── 8. Verify ──────────────────────────────────────────────────────────────
runs = list(client.list_runs(project_name="ragas-rag-eval", limit=10))
print(f"✅ LangSmith captured {len(runs)} run(s)")
print(f"🔗 https://smith.langchain.com/projects/ragas-rag-eval")

C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

Running RAGAS evaluation...


C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:74: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm=LangchainLLMWrapper(llm),
C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:75: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings=LangchainEmbeddingsWrapper(embeddings)
Evaluating:   5%|▌         | 1/20 [00:04<01:33,  4.95s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generat


── RAGAS Scores ──
{'faithfulness': 1.0000, 'answer_relevancy': 0.8518, 'context_precision': 1.0000, 'context_recall': 0.6000}

── Per-metric Breakdown ──
  faithfulness: 1.0000
  answer_relevancy: 0.8518
  context_precision: 1.0000
  context_recall: 0.6000

── Per-question Scores ──
   faithfulness  answer_relevancy  context_precision  context_recall
0           1.0          0.976414                1.0             0.0
1           1.0          0.959151                1.0             0.0
2           1.0          0.792460                1.0             1.0
3           1.0          0.679728                1.0             1.0
4           1.0          0.851113                1.0             1.0


C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:125: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.datetime.utcnow(),
C:\Users\sohai\AppData\Local\Temp\ipykernel_20240\3905963257.py:126: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time=datetime.datetime.utcnow()  # ✅ pass end_time in create_run directly



✅ Scores logged to LangSmith: {'faithfulness': 1.0, 'answer_relevancy': 0.8517732374794363, 'context_precision': 0.9999999999, 'context_recall': 0.6}
✅ LangSmith captured 10 run(s)
🔗 https://smith.langchain.com/projects/ragas-rag-eval
